# vector-db-bench — Analysis Notebook

Regenerates every chart from `results/demo/summary.parquet`.
Run via `make analysis` or `jupyter nbconvert --to notebook --execute analysis.ipynb`.

## 1  Setup

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

matplotlib.use("Agg")  # headless — no GUI required

RESULTS_DIR = Path("results/demo")
ASSETS_DIR = Path("assets")
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

# Colour palette keyed by adapter (db column)
ADAPTER_COLORS: dict[str, str] = {
    "pgvector": "#4C72B0",
    "qdrant": "#DD8452",
    "lancedb": "#55A868",
    "chroma": "#C44E52",
    "exact": "#8172B2",
}
DEFAULT_COLOR = "#999999"

print(f"Results dir : {RESULTS_DIR.resolve()}")
print(f"Assets dir  : {ASSETS_DIR.resolve()}")

Results dir : /Users/bisharamekhaeil/projects/vector-db-bench/results/demo
Assets dir  : /Users/bisharamekhaeil/projects/vector-db-bench/assets


## 2  Load

In [2]:
summary_path = RESULTS_DIR / "summary.parquet"
df = pd.read_parquet(summary_path)

print(f"Loaded {len(df)} rows from {summary_path}")
print()
print(df.head())
print()
print(df.dtypes)

Loaded 2 rows from results/demo/summary.parquet

         db                  label       params_hash  \
0  pgvector  pgvector:hnsw-default  3db35d3d9a027cdd   
1    qdrant    qdrant:hnsw-default  e44dff8986f66cb8   

                             params_json profile  n_passages  n_queries  dim  \
0  {"index": "hnsw", "metric": "cosine"}    warm        5000        100   64   
1                   {"metric": "cosine"}    warm        5000        100   64   

   ingest_s  ingest_throughput_vps  ...  latency_ms_p95  latency_ms_p99  \
0  2.728649            1832.408543  ...       18.968811       30.438136   
1  2.320294            2154.899490  ...       33.850897       40.313602   

   recall_at_k_mean  recall_at_k_p50  ndcg_at_k_mean  qps_estimate  \
0              0.88              1.0        0.926611     89.813993   
1              1.00              1.0        1.000000     50.242560   

   baseline_rss_bytes  index_rss_bytes  peak_rss_bytes  adapter_memory_bytes  
0           184508416    

In [3]:
# Derive columns required by the charts if absent from this parquet version.
#
# cost_per_million_queries_usd:
#   Estimated from QPS on c6i.xlarge @ $0.204 /hr (AWS on-demand, 2026-05).
#   cost = (1_000_000 queries / qps_estimate seconds) * ($/hr / 3600)
#   qps_estimate is already in the parquet.

HOURLY_RATE_USD = 0.204  # c6i.xlarge on-demand, us-east-1, 2026-05

if "cost_per_million_queries_usd" not in df.columns:
    df["cost_per_million_queries_usd"] = (
        (1_000_000 / df["qps_estimate"]) / 3600
    ) * HOURLY_RATE_USD
    print("Derived: cost_per_million_queries_usd")

# memory_mb: prefer adapter_memory_bytes (SDK-reported); fall back to
# peak_rss_bytes minus baseline_rss_bytes if present.
if "memory_mb" not in df.columns:
    if "adapter_memory_bytes" in df.columns and df["adapter_memory_bytes"].notna().any():
        mem_bytes = df["adapter_memory_bytes"]
    elif "peak_rss_bytes" in df.columns and "baseline_rss_bytes" in df.columns:
        mem_bytes = (df["peak_rss_bytes"] - df["baseline_rss_bytes"]).clip(lower=0)
    elif "peak_rss_bytes" in df.columns:
        mem_bytes = df["peak_rss_bytes"]
    else:
        mem_bytes = pd.Series([float("nan")] * len(df), index=df.index)
    df["memory_mb"] = mem_bytes / (1024 ** 2)
    print("Derived: memory_mb")

print()
print(df[["db", "label", "cost_per_million_queries_usd", "latency_ms_p95",
          "recall_at_k_mean", "ingest_s", "memory_mb"]].to_string(index=False))

Derived: cost_per_million_queries_usd
Derived: memory_mb

      db                 label  cost_per_million_queries_usd  latency_ms_p95  recall_at_k_mean  ingest_s  memory_mb
pgvector pgvector:hnsw-default                      0.630934       18.968811              0.88  2.728649        0.0
  qdrant   qdrant:hnsw-default                      1.127862       33.850897              1.00  2.320294        0.0


## 3  Cost vs latency Pareto

In [4]:
fig, ax = plt.subplots(figsize=(8, 5))

# Identify rows where cost is effectively zero/NaN (exact in-process baseline)
baseline_mask = (
    df["cost_per_million_queries_usd"].isna()
    | (df["cost_per_million_queries_usd"] == 0)
)

for _, row in df.iterrows():
    color = ADAPTER_COLORS.get(str(row["db"]), DEFAULT_COLOR)
    is_baseline = bool(baseline_mask.loc[row.name])
    if is_baseline:
        ax.scatter(
            row["cost_per_million_queries_usd"] if not pd.isna(row["cost_per_million_queries_usd"]) else 0,
            row["latency_ms_p95"],
            s=120,
            facecolors="none",
            edgecolors=color,
            linewidths=2,
            zorder=4,
        )
        ax.annotate(
            "in-process baseline",
            xy=(0, row["latency_ms_p95"]),
            xytext=(8, 0),
            textcoords="offset points",
            fontsize=8,
            va="center",
        )
    else:
        ax.scatter(
            row["cost_per_million_queries_usd"],
            row["latency_ms_p95"],
            s=100,
            color=color,
            label=str(row["db"]),
            zorder=4,
        )
        ax.annotate(
            str(row["db"]),
            xy=(row["cost_per_million_queries_usd"], row["latency_ms_p95"]),
            xytext=(6, 0),
            textcoords="offset points",
            fontsize=8,
            va="center",
        )

ax.set_xlabel("Cost per million queries (USD, est.)")
ax.set_ylabel("p95 latency (ms)")
ax.set_title("Cost vs latency Pareto")
ax.grid(True, alpha=0.3)

out_path = ASSETS_DIR / "analysis-cost-pareto.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")

Saved: assets/analysis-cost-pareto.png


## 4  Recall vs latency Pareto

In [5]:
fig, ax = plt.subplots(figsize=(8, 5))

recall_sorted = df.sort_values("latency_ms_p95")

# Pareto frontier: max recall seen so far as we increase latency
frontier_indices: list[int] = []
best_recall = -1.0
for idx, row in recall_sorted.iterrows():
    if float(row["recall_at_k_mean"]) > best_recall:
        best_recall = float(row["recall_at_k_mean"])
        frontier_indices.append(int(idx))

frontier_df = df.loc[frontier_indices]

for _, row in df.iterrows():
    color = ADAPTER_COLORS.get(str(row["db"]), DEFAULT_COLOR)
    on_frontier = row.name in frontier_indices
    ax.scatter(
        row["recall_at_k_mean"],
        row["latency_ms_p95"],
        s=140 if on_frontier else 80,
        color=color,
        marker="*" if on_frontier else "o",
        zorder=4,
        label=str(row["db"]),
    )
    ax.annotate(
        str(row["db"]),
        xy=(row["recall_at_k_mean"], row["latency_ms_p95"]),
        xytext=(6, 0),
        textcoords="offset points",
        fontsize=8,
        va="center",
    )

if len(frontier_df) > 1:
    ax.plot(
        frontier_df.sort_values("recall_at_k_mean")["recall_at_k_mean"],
        frontier_df.sort_values("recall_at_k_mean")["latency_ms_p95"],
        "k--",
        linewidth=1,
        alpha=0.4,
        label="frontier",
    )

ax.set_xlabel("Recall@k")
ax.set_ylabel("p95 latency (ms)")
ax.set_title("Recall vs latency Pareto (★ = frontier)")
ax.grid(True, alpha=0.3)

out_path = ASSETS_DIR / "analysis-recall-pareto.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")

Saved: assets/analysis-recall-pareto.png


## 5  Ingest throughput

In [6]:
fig, ax = plt.subplots(figsize=(7, 4))

adapters = df["db"].tolist()
values = df["ingest_s"].tolist()
colors = [ADAPTER_COLORS.get(str(a), DEFAULT_COLOR) for a in adapters]

bars = ax.bar(adapters, values, color=colors, edgecolor="white", linewidth=0.5)
for bar, val in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(values) * 0.01,
        f"{val:.1f}s",
        ha="center",
        va="bottom",
        fontsize=9,
    )

ax.set_xlabel("Adapter")
ax.set_ylabel("Ingest time (seconds)")
ax.set_title("Ingest throughput — wall-clock seconds")
ax.grid(axis="y", alpha=0.3)

out_path = ASSETS_DIR / "analysis-ingest.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out_path}")

Saved: assets/analysis-ingest.png


## 6  Memory footprint

In [7]:
if "memory_mb" in df.columns and df["memory_mb"].notna().any() and (df["memory_mb"] > 0).any():
    fig, ax = plt.subplots(figsize=(7, 4))

    adapters = df["db"].tolist()
    mem_vals = df["memory_mb"].fillna(0).tolist()
    colors = [ADAPTER_COLORS.get(str(a), DEFAULT_COLOR) for a in adapters]

    bars = ax.bar(adapters, mem_vals, color=colors, edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, mem_vals):
        if val > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(mem_vals) * 0.01,
                f"{val:.1f} MB",
                ha="center",
                va="bottom",
                fontsize=9,
            )
        else:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                max(mem_vals) * 0.02,
                "N/A",
                ha="center",
                va="bottom",
                fontsize=9,
                color="#888",
            )

    ax.set_xlabel("Adapter")
    ax.set_ylabel("Memory footprint (MB)")
    ax.set_title("Memory footprint per adapter")
    ax.grid(axis="y", alpha=0.3)

    out_path = ASSETS_DIR / "analysis-memory.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")
else:
    print("memory_mb column absent or all-zero — skipping memory chart")

memory_mb column absent or all-zero — skipping memory chart


## 7  Headline summary table

In [8]:
headline_cols = {
    "db": "Adapter",
    "ingest_s": "Ingest (s)",
    "latency_ms_p95": "p95 (ms)",
    "recall_at_k_mean": "Recall@k",
    "qps_estimate": "QPS (est.)",
    "cost_per_million_queries_usd": "$/M-queries",
    "memory_mb": "Memory (MB)",
}

present_cols = {k: v for k, v in headline_cols.items() if k in df.columns}
summary_table = df[list(present_cols)].copy().rename(columns=present_cols)

# Format numeric columns
fmt = {
    "Ingest (s)": "{:.2f}",
    "p95 (ms)": "{:.1f}",
    "Recall@k": "{:.3f}",
    "QPS (est.)": "{:.0f}",
    "$/M-queries": "{:.4f}",
    "Memory (MB)": "{:.1f}",
}
for col, fmtstr in fmt.items():
    if col in summary_table.columns:
        summary_table[col] = summary_table[col].apply(
            lambda x: fmtstr.format(x) if pd.notna(x) else "—"
        )

print(summary_table.to_string(index=False))

 Adapter Ingest (s) p95 (ms) Recall@k QPS (est.) $/M-queries Memory (MB)
pgvector       2.73     19.0    0.880         90      0.6309         0.0
  qdrant       2.32     33.9    1.000         50      1.1279         0.0
